# CIFAR pilot — Kaggle T4×2
Enable Internet, T4×2 and the `github_token` secret; attach real CIFAR-100.
Two queues run six bounded real-data pilots, aiming for 15–20 minutes total.
Each uses two epochs, ≤20 training batches/epoch, ≤5 validation/test batches,
and one real measurement (rank 4, two null repeats, CG 200).
B3 runs two tasks so the final measurement exercises iCaRL KD.
B2 fetches and hash-verifies its teacher and reports a bounded teacher gate.
B4 trains then measures the epoch-2 checkpoint in the same process.
All gates are PILOT. Each job has its own log and failures leave both queues running.
No archives or full training runs. Reduced diagnostics can underestimate full costs.
Regenerate with the eventual published commit containing this implementation before Kaggle.


In [ ]:
REPO_COMMIT = "9fde1cb0bf8dfd68319e80310fe9755f624b9ab7"
PILOT_BATCHES = 20
MU_REL_DEFAULT = 1e-2
LANES = [[{'name': 'b3', 'module': 'experiments.stage_b3', 'args': []}, {'name': 'b1_resnet8x4', 'module': 'experiments.stage_b12', 'args': ['b1', '--student', 'resnet8x4']}, {'name': 'b1_wrn_16_2', 'module': 'experiments.stage_b12', 'args': ['b1', '--student', 'wrn_16_2']}], [{'name': 'b2_resnet8x4', 'module': 'experiments.stage_b12', 'args': ['b2', '--student', 'resnet8x4']}, {'name': 'b2_wrn_16_2', 'module': 'experiments.stage_b12', 'args': ['b2', '--student', 'wrn_16_2']}, {'name': 'b4', 'module': 'experiments.stage_b4', 'args': ['train']}]]


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/vna_cifar_pilot")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)

if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "diagnostics/signal_curve.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO), str(GROMO_SRC), str(REFERENCE)]
print("Official TEST verified:", DATA_ROOT / "cifar-100-python/test")

MU_REL_DEFAULT = 1e-2
MU_SOURCE = "fallback:1e-2"
inputs = Path("/kaggle/input")
candidates = sorted(inputs.glob("**/stageA/summary.json"))
candidates += [p for p in sorted(inputs.rglob("summary.json")) if p not in candidates]
for path in candidates:
    try:
        summary = json.loads(path.read_text())
    except (ValueError, OSError): continue
    if "mu_rel_default" in summary:
        MU_REL_DEFAULT = summary["mu_rel_default"]; MU_SOURCE = str(path); break
else:
    print("*** WARNING: NO ATTACHED STAGE A MU; USING 1e-2 ***", flush=True)
print("mu_rel_default:", MU_REL_DEFAULT, "mu_source:", MU_SOURCE)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
PILOT = RUN_ROOT / "outputs" / "pilot"
PILOT.mkdir(parents=True, exist_ok=True)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python"}
def pilot_lane(gpu, jobs):
    results = []
    for job in jobs:
        out = PILOT / job["name"]
        # Fresh outputs ensure a failed rerun cannot reuse a stale summary.
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True)
        log_path = PILOT / (job["name"] + ".log")
        cmd = [sys.executable, "-u", "-m", job["module"], *job["args"],
               "--pilot", "--pilot-batches", str(PILOT_BATCHES), "--out", str(out),
               "--device", "cuda:0", "--workers", "2", "--data-dir", str(DATA_ROOT),
               "--mu-rel-default", str(MU_REL_DEFAULT), "--mu-source", MU_SOURCE]
        try:
            with log_path.open("w", buffering=1) as log:
                completed = subprocess.run(cmd, cwd=REPO, env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)},
                                           stdout=log, stderr=subprocess.STDOUT, check=False)
            ok = completed.returncode == 0 and (out / "summary.json").exists()
            error = "" if ok else f"exit {completed.returncode}"
        except Exception as exc:
            ok, error = False, str(exc)
            with log_path.open("a") as log: log.write(error + "\n")
        results.append({"setting": job["name"], "ok": ok, "error": error, "log": str(log_path)})
        print(job["name"], "OK" if ok else "FAILED", flush=True)
    return results
with ThreadPoolExecutor(max_workers=2) as pool:
    RESULTS = [result for lane in pool.map(lambda item: pilot_lane(*item), enumerate(LANES)) for result in lane]
(PILOT / "jobs.json").write_text(json.dumps(RESULTS, indent=2))


In [ ]:
import pandas as pd
rows = []
for result in RESULTS:
    name = result["setting"]
    path = PILOT / name / "summary.json"
    summary = json.loads(path.read_text()) if path.exists() else {}
    status = "OK" if result["ok"] else "FAILED: " + result["error"] + "\n" + "\n".join(Path(result["log"]).read_text().splitlines()[-20:])
    rows.append({"setting": name, "status": status,
                 "epoch seconds": summary.get("epoch_seconds"),
                 "measurement seconds": summary.get("seconds_per_measurement_point"),
                 "peak memory (bytes)": summary.get("peak_gpu_memory_bytes"),
                 "projected full GPU-hours": summary.get("projected_full_gpu_hours"),
                 "gate keys present": summary.get("gate_keys_present", False)})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows))
for notebook, lanes in {"nb2": (("b3",), ("b2_resnet8x4", "b4")),
                        "nb3": (("b1_resnet8x4", "b1_wrn_16_2"), ("b2_wrn_16_2",))}.items():
    for gpu, names in enumerate(lanes):
        selected = [r for r in rows if r["setting"] in names]
        total = sum(r["projected full GPU-hours"] or 0 for r in selected)
        if notebook == "nb2":
            b4_path = PILOT / "b4/summary.json"
            if b4_path.exists():
                b4 = json.loads(b4_path.read_text())
                measure_hours = 12 * b4.get("seconds_per_measurement_point", 0) / 3600
                total += measure_hours if gpu == 0 else -measure_hours
        complete = all(r["status"] == "OK" for r in selected)
        print(f"{notebook} GPU{gpu}: {total:.3f} projected GPU-hours" +
              ("" if complete else " (partial; failed jobs omitted)"))
